In [8]:
import os

In [9]:
pwd

'/Users/Alex/DS projects/mySummerizer'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    

In [14]:
from mySummarizer.constants import *
from mySummarizer.utils.common import read_yaml, create_directories

In [86]:
class ConfiguartionManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(str(config_filepath))
        self.params = read_yaml(str(params_filepath))
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        try:
            config = self.config.data_ingestion
        except KeyError:
            raise ValueError("The config YAML file is missing the 'data_ingestion' key. Please check the file structure.")

        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [87]:
import os
import urllib.request as request
import zipfile
from mySummarizer.logging import logger
from mySummarizer.utils.common import get_size

In [88]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"Downloaded file: {filename} with the following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_file(self):
        """
        Unzip the downloaded file.
        Extracts the zip file to the specified directory.
        Functions returns None.

        """

        unzip_dir = self.config.unzip_dir
        os.makedirs(unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

In [92]:
try:
    config = ConfiguartionManager()
    data_ingestion_config = config.get_data_ingestion_config()
    print(data_ingestion_config)
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_file()
except Exception as e:
    raise e

[2026-03-16 01:52:08,402]: INFO: common: Successfully read YAML file: config/config.yaml
[2026-03-16 01:52:08,440]: INFO: common: Successfully read YAML file: params.yaml
[2026-03-16 01:52:08,441]: INFO: common: Created directory at: artifacts
[2026-03-16 01:52:08,442]: INFO: common: Created directory at: artifacts/data_ingestion
DataIngestionConfig(root_dir='artifacts/data_ingestion', source_URL='https://github.com/ashfaqahasan55/mySummerizer/raw/refs/heads/main/samsum_data.zip', local_data_file='artifacts/data_ingestion/samsum_data.zip', unzip_dir='artifacts/data_ingestion')
[2026-03-16 01:52:09,968]: INFO: 897069027: Downloaded file: artifacts/data_ingestion/samsum_data.zip with the following info: 
Connection: close
Content-Length: 4009553
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "300346fde828401c7ccfc05cbcf61c9c91f493fa519d16e3e798140055bd9943"
Strict-Transport-Security: max-age=3